# Using the live flywheel

This notebook is only about the **runtime**: readiness → live decisions → monitoring → containment → approval → repair → retraction. It uses a small diabetes regressor so every step runs in a few seconds.

The law you will see enforced:

> **Tightening is always allowed. Loosening requires an external signature.**

- `readiness()` is a one-shot deployment gate. It mutates nothing.
- `LiveSession` journals decisions, observes delayed outcomes (ACI only), and runs deterministic sequential monitors.
- `contain()` is unconditional and may only turn a release into an ask.
- `uncontain()` requires an approval the session can verify but cannot sign.
- `ChangeSet` promotion is evidence-gated: replay stored facts, compare the pair, refuse loosening without an approval, create a new immutable version.
- `retract()` lists the certificates issued under a faulty version for human triage.

## 1. Setup: a fitted model, a declared policy, an external hazard

The policy is the only disposer. The learner only emits facts (`pred`, `lo`, `hi`).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from mjolnir_ml import (ASK, DENY, RELEASE, ApprovalAuthority, ChangeSet,
                        DecisionModel, DecisionPolicyBase, DetectorSpec,
                        HazardLedger, Journal, LiveSession, dsl, readiness, rule)

pd.set_option("display.width", 140)

@dsl
class FlywheelPolicy(DecisionPolicyBase):
    facts = ("age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6")
    assert lambda bmi: bmi > -1.0, "bmi_in_range"
    assert rule("release_safe_band",
                when=lambda hi: hi <= 240.0, verdict=RELEASE)
    assert rule("deny_high_band", when=lambda lo: lo > 240.0, verdict=DENY)
    assert rule("review_widening",
                when=lambda lo, hi: lo <= 240.0 and hi > 240.0, verdict=ASK)

@dsl
class FlywheelPolicyV2(DecisionPolicyBase):
    facts = ("age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6")
    assert lambda bmi: bmi > -1.0, "bmi_in_range"
    assert rule("deny_age_above_spec", when=lambda age: age > 0.10, verdict=DENY)
    assert rule("release_safe_band",
                when=lambda hi: hi <= 240.0, verdict=RELEASE)
    assert rule("deny_high_band", when=lambda lo: lo > 240.0, verdict=DENY)
    assert rule("review_widening",
                when=lambda lo, hi: lo <= 240.0 and hi > 240.0, verdict=ASK)

HAZARD_AGE = 0.10
ledger = (HazardLedger()
          .add("age_above_spec", lambda r: r["age"] > HAZARD_AGE,
               "underwriting spec §2 (age band)"))
ledger.as_dict()

{'schema': 'mjolnir.ml.hazard-ledger/1',
 'version': 'hazard-ledger@1',
 'active': {'age_above_spec': {'provenance': 'underwriting spec §2 (age band)',
   'observable': True}},
 'retired': {}}

In [2]:
data = load_diabetes()
names = tuple(data.feature_names)
X, y = data.data, data.target
Xtr, Xrest, ytr, yrest = train_test_split(X, y, test_size=0.4, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5, random_state=0)

rows = lambda A: [dict(zip(names, map(float, x))) for x in A]
forest = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
dm = (DecisionModel(forest, FlywheelPolicy, features=names)
      .fit_regime(rows(Xtr))
      .calibrate(rows(Xcal), ycal, alpha=0.10, adaptive=True,
                 gamma=0.10, window=150))
print("features:", len(names), "| calibrated:", dm.kappa is not None,
      "| alpha:", dm.kappa.alpha)

features: 10 | calibrated: True | alpha: 0.1


## 2. Readiness — the one-shot gate

It asks one question: *may this (model, policy, calibration) triple deploy?* It needs an oracle (`safe`, `decidable`), so refusing to ask is not a pass. Nothing is mutated.

In [3]:
ready_rows = rows(Xte)[:20]
ready_y = list(yte)[:20]
oracle = lambda row, yy: (float(yy) <= 240.0 and not ledger.hits(row), True)

rep = readiness(dm, ready_rows,
                safe=[float(v) <= 240.0 for v in ready_y],
                decidable=[True] * len(ready_y),
                ledger=ledger)
print("approved:", rep.approved, "| readiness_id:", rep.readiness_id[:26], "…")
pd.DataFrame([g.as_dict() for g in rep.gates])

approved: True | readiness_id: sha256:5d2672c6020a6a89e41 …


,name,passed,value,threshold,detail
0,calibration_present,True,True,required,unknown is not zero: a policy needing conforma...
1,overclaim,True,0,0,claims on oracle-undecidable rows
2,precision,True,1.0,0.95,certified claims / claims
3,recall,True,0.4,0.0,certified claims / decidable rows
4,coverage,True,0.4,0.0,claims / rows
5,abstain_rate,True,0.0,1.0,a deployment that abstains on everything is no...


## 3. Open a live session

The journal is append-only and hash-chained. `decide` writes the row, facts, verdict and digests; `observe` joins the delayed outcome and updates **only** κ (ACI). Detectors are declared sequential CUSUMs with a minimum detectable effect. Nothing here is a background thread: `tick()` is explicit and deterministic.

In [4]:
detectors = (
    DetectorSpec("coverage_low", "index", "coverage", 0.90, slack=0.02,
                 threshold=4.0, min_n=20, mde=0.05),
    DetectorSpec("abstain_high", "contract", "abstain", 0.20, slack=0.01,
                 threshold=5.0, min_n=20, mde=0.05),
    DetectorSpec("hazard_release", "contract", "hazard_release", 0.0,
                 slack=0.0, threshold=0.0, min_n=1, mde=0.0),
)

authority = ApprovalAuthority(b"notebook-approval")   # signing key lives here
live = LiveSession(dm, journal=Journal(), ledger=ledger, detectors=detectors,
                   approval_verifier=authority.verify)  # verifier only
print("versions:", live.versions)

versions: [{'policy_digest': 'sha256:b42a274ad237af44e142c0b5bc29aebef4c22b1deeca16d9c470f8f2cf724050', 'model_digest': 'sha256:bf42fa8e1c359011211d7048d5874479987fa19c2e44d776011a5ffa42c04b24', 'calibration_digest': 'sha256:0df43d480960e5390e8dbf2e7ec5c3865ced2def10d0e9dc5a04b3641134b2ac'}]


## 4. Batches arrive — decide, observe, monitor, act

Each batch is decided, outcomes are joined, then `tick()` runs the declared detectors. `auto_act` has exactly one automatic action: **contain** on a released external hazard. Warnings never act; the human review queue collects abstentions.

The stream is simulated: drift starts at batch 3, and a hazardous low-risk row is injected at batch 3 (age above the underwriting band) and batch 4.

In [5]:
stream_rows = rows(Xte)[20:]
stream_y = [float(v) for v in list(yte)[20:]]
injection = {**stream_rows[0], "age": 0.105, "bmi": -0.05, "bp": -0.09}

batches = [list(range(i, min(i + 10, len(stream_rows))))
           for i in range(0, len(stream_rows), 10)][:4]
history = []

for b, idxs in enumerate(batches, start=1):
    for j, i in enumerate(idxs):
        row, yy = dict(stream_rows[i]), stream_y[i]
        if b >= 3:                              # drift + injected hazard
            yy += 80.0
            row["bp"] += 0.03
            row["bmi"] += 0.02
        if (b == 3 and j == 0) or (b == 4 and j == 0):
            row, yy = dict(injection), 80.0
        d = live.decide(row, batch=b)
        live.observe(d.certificate_id, yy)

    dets = live.tick()
    actions = live.auto_act(dets, batch=b)
    st = live.status(window_batches=1)
    history.append({"batch": b,
                    **st["verdicts"],
                    "coverage": st["coverage"],
                    "mae": st["mae"],
                    "hazards": st["released_hazards"],
                    "contained": bool(st["contained"]),
                    "alarms": ", ".join(d.name for d in dets if d.alarm)})
    print(f"batch {b}: {st['verdicts']} | coverage={st['coverage']:.2f} "
          f"| contained={bool(st['contained'])}")

pd.DataFrame(history)

batch 1: {'release': 3, 'ask': 7, 'deny': 0, 'abstain': 0} | coverage=0.80 | contained=False


batch 2: {'release': 3, 'ask': 7, 'deny': 0, 'abstain': 0} | coverage=1.00 | contained=False


batch 3: {'release': 3, 'ask': 6, 'deny': 0, 'abstain': 1} | coverage=0.89 | contained=True


batch 4: {'release': 0, 'ask': 10, 'deny': 0, 'abstain': 0} | coverage=0.80 | contained=True


,batch,release,ask,deny,abstain,coverage,mae,hazards,contained,alarms
0,1,3,7,0,0,0.800000,60.433500,0,False,
1,2,3,7,0,0,1.000000,40.018000,0,False,
2,3,3,6,0,1,0.888889,50.817222,1,True,hazard_release
3,4,0,10,0,0,0.800000,61.413000,0,True,"abstain_high, hazard_release"


## 5. What the flywheel did

At the hazard release the session contains itself: releases become asks. The action is unconditional and tighten-only. The human queue is where abstentions go for review.

In [6]:
print("containment:", live.containment.as_dict() if live.containment else None)
for item in live.status()["review_queue"][:3]:
    print(f"- batch {item['batch']}: {item['reason']} — {item['say'][:90]}")

containment: {'containment_id': 'sha256:d3bef20e6a80d0d29205975f82b3c1c95b0921d2bf9775d99617fbfb7e3dee44', 'reason': '1 release(s) on external hazard rows', 'batch': 3, 'at': '2026-10-25T17:17:00Z'}
- batch 3: out_of_regime:bp — cannot decide: bp=0.137944 is outside recorded training bounds [-0.108956, 0.132044]


## 6. Repair as a ChangeSet — replay, Pareto, promote

A policy change is evaluated by replaying the **stored facts** through `Policy.dispose_facts` — no model. The oracle encodes the external hazard: a release on a hazard row is not correct, even when the outcome happens to be safe.

Adding the missing deny rule only removes releases (`may_loosen=False`), so it is promoted without approval and becomes a new immutable version.

In [7]:
old_policy = live.model.policy.digest
change = ChangeSet(lever="contract", reason="add the declared age-band hazard",
                   policy=FlywheelPolicyV2.policy(), name="repair_age_band")

ev = live.evaluate(change, oracle=oracle, ledger=ledger)
print("gate pass:", ev.passed, "| may_loosen:", ev.may_loosen,
      "| hazards:", ev.before["released_hazards"], "->", ev.after["released_hazards"])
pd.DataFrame([g.as_dict() for g in ev.gates])

gate pass: True | may_loosen: False | hazards: 2 -> 0


,name,passed,before,after
0,hazard_releases_not_increased,True,2.000000,0.000000
1,precision_non_regression,True,0.769231,0.923077
2,recall_non_regression,True,0.250000,0.300000
3,improves_something,True,NaN,NaN


In [8]:
prom = live.promote(change, oracle=oracle, ledger=ledger)
print("promoted:", prom.approved, "|", prom.reason)
print("new policy:", live.model.policy.digest[:26], "…")
print("versions:", len(live.versions))

retraction = live.retract(policy_digest=old_policy,
                          reason="decisions under v1 (missing age-band rule)")
print("retracted certificates:", len(retraction.certificate_ids))

promoted: True | promoted
new policy: sha256:e6a05910567603d35f8 …
versions: 2
retracted certificates: 40


## 7. Lifting containment is a loosening — it needs an external signature

The session never holds the signing key. It receives only `authority.verify`. The approval is bound to the containment digest, so it cannot be reused for another containment.

In [9]:
approval = authority.sign(live.containment.change_digest, approver="dr-owner")
live.uncontain(approval, reason="repair promoted; the hazard rule now denies the band")
print("contained now:", live.containment)
print("hazard after repair:", live.model.dispose(injection).verdict)

contained now: None
hazard after repair: deny


## 8. Final state and journal integrity

All decisions, outcomes, containment, approvals, changes and retractions are in the hash-chained journal. Replaying it is what makes the flywheel auditable.

In [10]:
final = live.status()
print("journal verified:", live.journal.verify(), "| entries:", len(live.journal))
print("final verdicts:", final["verdicts"])
print("final coverage:", final["coverage"], "| versions:", len(live.versions))
print("change records:", len(live.journal.payloads("change")))
print("retraction records:", len(live.journal.payloads("retraction")))

journal verified: True | entries: 84
final verdicts: {'release': 9, 'ask': 30, 'deny': 0, 'abstain': 1}
final coverage: 0.8717948717948718 | versions: 2
change records: 1
retraction records: 1


## 9. The rules to take away

- **The learner proposes facts; the policy disposes.** A model swap changes `model_digest`, never `policy_digest`.
- **κ may learn.** `observe` updates ACI; it cannot touch a rule. Recalibration is a ChangeSet, not a silent side effect.
- **Containment is free; loosening is signed.** `contain` is unconditional and only tightens. `uncontain` and any change that adds a release need an external approval.
- **Promotion is evidence-gated.** Replay stored facts, compare the pair with a Pareto gate, and promote a **new version**; keep the old one for rollback.
- **Retract, don't delete.** Certificates issued under a faulty version are listed for human triage.
- **Monitors act only on critical evidence.** Warnings go to humans; only a released external hazard triggers an automatic action.
- **Determinism is the audit.** `tick()` from the same journal yields the same alarms; the journal is hash-chained.